In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
import copy
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix, roc_curve
)
from sklearn.linear_model import LogisticRegression

In [3]:
# ─────────────────────────────────────────────
# 0. SEED & DEVICE
# ─────────────────────────────────────────────
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)
torch.cuda.manual_seed_all(RANDOM_STATE)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cpu


In [4]:
# ─────────────────────────────────────────────
# 1. PATHS
# ─────────────────────────────────────────────
BASE_DIR = "/content/drive/MyDrive/data after deg"
RESULT_DIR = "/content/drive/MyDrive/results_mlp_lasso_sfs"
os.makedirs(RESULT_DIR, exist_ok=True)

X_TRAIN_PATH = os.path.join(BASE_DIR, "X_train_vst.csv")
X_TEST_PATH  = os.path.join(BASE_DIR, "X_test_vst.csv")
Y_TRAIN_PATH = os.path.join(BASE_DIR, "y_train_lung.csv")
Y_TEST_PATH  = os.path.join(BASE_DIR, "y_test_lung.csv")

In [5]:
# ─────────────────────────────────────────────
# 2. LOAD & CLEAN
# ─────────────────────────────────────────────
def clean_feature_df(df):
    df = df.copy()
    return df.loc[:, ~df.columns.str.contains("^Unnamed")]

def clean_label_df(df):
    df = df.copy()
    df = df.loc[:, ~df.columns.str.contains("^Unnamed")]
    return df.iloc[:, -1].astype(int)

X_train_full = clean_feature_df(pd.read_csv(X_TRAIN_PATH, low_memory=False))
X_test       = clean_feature_df(pd.read_csv(X_TEST_PATH,  low_memory=False))
y_train_full = clean_label_df(pd.read_csv(Y_TRAIN_PATH,  low_memory=False))
y_test       = clean_label_df(pd.read_csv(Y_TEST_PATH,   low_memory=False))

In [6]:
# ─────────────────────────────────────────────
# 2.1 LASSO FEATURE SELECTION
# ─────────────────────────────────────────────
# LASSO dùng Logistic Regression với L1 penalty để lấy độ quan trọng gene.
K = 208

lasso_scaler = StandardScaler()
X_train_lasso_input = lasso_scaler.fit_transform(X_train_full)

lasso = LogisticRegression(
    penalty="l1",
    solver="liblinear",
    C=0.1,
    random_state=RANDOM_STATE,
    max_iter=5000
)
lasso.fit(X_train_lasso_input, y_train_full)

coef_abs = np.abs(lasso.coef_).ravel()
lasso_scores = pd.DataFrame({
    "Gene": X_train_full.columns,
    "LASSO_abs_coef": coef_abs
}).sort_values("LASSO_abs_coef", ascending=False)

# Nếu số gene có hệ số khác 0 ít hơn K thì vẫn lấy top K theo |coef|.
gene_names = lasso_scores.head(K)["Gene"].tolist()

lasso_scores.to_csv(
    os.path.join(RESULT_DIR, "lasso_gene_scores_mlp.csv"),
    index=False
)

X_train_lasso = X_train_full[gene_names].values
X_test_lasso  = X_test[gene_names].values

X_train_full = pd.DataFrame(X_train_lasso, columns=gene_names)
X_test = pd.DataFrame(X_test_lasso, columns=gene_names)

print(f"LASSO selected genes: {len(gene_names)}")
print("Top 20 LASSO genes:")
print(gene_names[:20])

LASSO selected genes: 208
Top 20 LASSO genes:
['ENSG00000227165', 'ENSG00000133742', 'ENSG00000251562', 'ENSG00000142089', 'ENSG00000069535', 'ENSG00000162894', 'ENSG00000184613', 'ENSG00000103316', 'ENSG00000249307', 'ENSG00000151789', 'ENSG00000048052', 'ENSG00000115457', 'ENSG00000143546', 'ENSG00000149243', 'ENSG00000130948', 'ENSG00000163359', 'ENSG00000173083', 'ENSG00000185909', 'ENSG00000142856', 'ENSG00000103512']


In [7]:
# ─────────────────────────────────────────────
# 3. SCALE  (fit on full train, transform test)
# ─────────────────────────────────────────────
scaler = StandardScaler()
X_train_full = pd.DataFrame(
    scaler.fit_transform(X_train_full), columns=gene_names
)
X_test = pd.DataFrame(
    scaler.transform(X_test), columns=gene_names
)

In [8]:
# ─────────────────────────────────────────────
# 4. TRAIN / VAL SPLIT
# ─────────────────────────────────────────────
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full,
    test_size=0.2, stratify=y_train_full,
    random_state=RANDOM_STATE
)

In [9]:
# ─────────────────────────────────────────────
# 5. DATASET & HELPERS
# ─────────────────────────────────────────────
class GeneExpressionDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X.values, dtype=torch.float32)
        self.y = torch.tensor(y.values, dtype=torch.float32)
    def __len__(self): return len(self.X)
    def __getitem__(self, idx): return self.X[idx], self.y[idx]


def make_loaders(X_tr, X_vl, X_te, y_tr, y_vl, y_te, batch_size=32):
    return (
        DataLoader(GeneExpressionDataset(X_tr, y_tr), batch_size=batch_size, shuffle=True),
        DataLoader(GeneExpressionDataset(X_vl, y_vl), batch_size=batch_size, shuffle=False),
        DataLoader(GeneExpressionDataset(X_te, y_te), batch_size=batch_size, shuffle=False),
    )


def run_epoch(model, loader, criterion, optimizer=None):
    model.train() if optimizer else model.eval()
    total_loss, all_probs, all_labels = 0, [], []

    for X_b, y_b in loader:
        X_b, y_b = X_b.to(device), y_b.to(device)
        with torch.set_grad_enabled(optimizer is not None):
            logits = model(X_b)
            loss   = criterion(logits, y_b)
            if optimizer:
                optimizer.zero_grad(); loss.backward(); optimizer.step()

        all_probs.extend(torch.sigmoid(logits).detach().cpu().numpy())
        all_labels.extend(y_b.detach().cpu().numpy())
        total_loss += loss.item() * len(X_b)

    return total_loss / len(loader.dataset), np.array(all_labels), np.array(all_probs)


def compute_metrics(y_true, y_prob, threshold=0.5):
    y_pred = (y_prob >= threshold).astype(int)
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel() if cm.shape == (2, 2) else (0, 0, 0, 0)
    sn = recall_score(y_true, y_pred, zero_division=0)
    sp = tn / (tn + fp) if (tn + fp) > 0 else 0
    return {
        "ACC":       accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Sn":        sn,
        "Sp":        sp,
        "F1":        f1_score(y_true, y_pred, zero_division=0),
        "AUC":       roc_auc_score(y_true, y_prob) if len(np.unique(y_true)) > 1 else 0.0,
        "BA":        (sn + sp) / 2,
    }


def fmt(v): return f"{v * 100:.2f}%"

In [10]:
# ─────────────────────────────────────────────
# 6. MODEL
# ─────────────────────────────────────────────
class MLPDropoutBN(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 512), nn.BatchNorm1d(512), nn.ReLU(True), nn.Dropout(0.3),
            nn.Linear(512, 256),       nn.BatchNorm1d(256), nn.ReLU(True), nn.Dropout(0.3),
            nn.Linear(256, 128),       nn.BatchNorm1d(128), nn.ReLU(True), nn.Dropout(0.2),
            nn.Linear(128, 1)
        )
    def forward(self, x): return self.net(x).squeeze(-1)

In [11]:
# ─────────────────────────────────────────────
# 7. HYPERPARAMS
# ─────────────────────────────────────────────
BATCH_SIZE  = 32
EPOCHS      = 100
PATIENCE    = 15
START_GENE  = 10   # mirror Document 2: ranks[10:]
COLS = ['Gene', 'N_Genes', 'ACC', 'Sn', 'Precision', 'Sp', 'F1', 'AUC', 'BA', 'Threshold']

result_val  = []
result_test = []

pos_count  = y_train.sum()
neg_count  = len(y_train) - pos_count
pos_weight = torch.tensor([neg_count / pos_count], dtype=torch.float32).to(device)

In [12]:
# ─────────────────────────────────────────────
# 8. SFS LOOP
# ─────────────────────────────────────────────
for i in range(START_GENE, len(gene_names)):

    selected = gene_names[:i + 1]          # top-(i+1) LASSO-ranked genes
    current_gene = gene_names[i]           # the gene added this round

    train_loader, val_loader, test_loader = make_loaders(
        X_train[selected], X_val[selected], X_test[selected],
        y_train, y_val, y_test,
        batch_size=BATCH_SIZE
    )

    # ── Build fresh model ──────────────────────
    model     = MLPDropoutBN(input_dim=i + 1).to(device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

    # ── Train with early stopping ──────────────
    best_val_auc    = -1
    wait            = 0
    best_weights    = None          # store in RAM, not disk

    for epoch in range(1, EPOCHS + 1):
        run_epoch(model, train_loader, criterion, optimizer)

        _, y_v, p_v = run_epoch(model, val_loader, criterion)
        val_auc = compute_metrics(y_v, p_v)["AUC"]

        if val_auc > best_val_auc:
            best_val_auc = val_auc
            wait         = 0
            best_weights = copy.deepcopy(model.state_dict())   # RAM save
        else:
            wait += 1

        if wait >= PATIENCE:
            break

    # ── Load best weights ──────────────────────
    model.load_state_dict(best_weights)
    model.eval()

    # ── Threshold selection on validation ─────
    _, y_val_true, y_val_prob = run_epoch(model, val_loader, criterion)
    fpr, tpr, thresholds = roc_curve(y_val_true, y_val_prob)
    best_threshold = thresholds[np.argmax(tpr - fpr)]

    val_metrics = compute_metrics(y_val_true, y_val_prob, threshold=best_threshold)

    # ── Test evaluation ────────────────────────
    _, y_test_true, y_test_prob = run_epoch(model, test_loader, criterion)
    test_metrics = compute_metrics(y_test_true, y_test_prob, threshold=best_threshold)

    # ── Log ───────────────────────────────────
    print(
        f"Genes={i+1:3d} | added={current_gene:<20s} | "
        f"val_AUC={val_metrics['AUC']:.4f} | "
        f"test_AUC={test_metrics['AUC']:.4f} | "
        f"thr={best_threshold:.4f}"
    )

    def row(m): return [
        current_gene, i + 1,
        fmt(m["ACC"]), fmt(m["Sn"]), fmt(m["Precision"]),
        fmt(m["Sp"]),  fmt(m["F1"]), fmt(m["AUC"]), fmt(m["BA"]),
        f"{best_threshold:.4f}"
    ]

    result_val.append(row(val_metrics))
    result_test.append(row(test_metrics))

    # ── Save incrementally (safe if interrupted) ──
    pd.DataFrame(result_val,  columns=COLS).to_csv(
        os.path.join(RESULT_DIR, "mlp_bn_sfs_val.csv"),  index=False)
    pd.DataFrame(result_test, columns=COLS).to_csv(
        os.path.join(RESULT_DIR, "mlp_bn_sfs_test.csv"), index=False)

print("\nDone. Results saved to:", RESULT_DIR)
df_test = pd.read_csv(os.path.join(RESULT_DIR, "mlp_bn_sfs_test.csv"))

df_test["AUC_num"] = df_test["AUC"].str.replace("%", "").astype(float)

best = df_test.loc[df_test["AUC_num"].idxmax()]

print("\n" + "="*60)
print("BEST MLP BN SFS TEST RESULT - CHOSEN BY AUC")
print("="*60)

print("Method      : LASSO + MLP BatchNorm SFS")
print("Gene_number :", best["N_Genes"])
print("ACC         :", best["ACC"])
print("Recall/Sn   :", best["Sn"])
print("Precision   :", best["Precision"])
print("F1          :", best["F1"])
print("AUC         :", best["AUC"])
print("BA          :", best["BA"])

print("\nCOPY TO TABLE:")
print(
    f"LASSO + MLP BatchNorm SFS | "
    f"{best['ACC']} | {best['Sn']} | {best['Precision']} | "
    f"{best['F1']} | {best['AUC']} | {best['BA']} | {best['N_Genes']}"
)

Genes= 11 | added=ENSG00000048052      | val_AUC=0.9595 | test_AUC=0.9420 | thr=0.4896
Genes= 12 | added=ENSG00000115457      | val_AUC=0.9629 | test_AUC=0.9380 | thr=0.4416
Genes= 13 | added=ENSG00000143546      | val_AUC=0.9665 | test_AUC=0.9397 | thr=0.4838
Genes= 14 | added=ENSG00000149243      | val_AUC=0.9721 | test_AUC=0.9431 | thr=0.4424
Genes= 15 | added=ENSG00000130948      | val_AUC=0.9686 | test_AUC=0.9364 | thr=0.5085
Genes= 16 | added=ENSG00000163359      | val_AUC=0.9740 | test_AUC=0.9407 | thr=0.5154
Genes= 17 | added=ENSG00000173083      | val_AUC=0.9727 | test_AUC=0.9409 | thr=0.3783
Genes= 18 | added=ENSG00000185909      | val_AUC=0.9757 | test_AUC=0.9381 | thr=0.6728
Genes= 19 | added=ENSG00000142856      | val_AUC=0.9778 | test_AUC=0.9419 | thr=0.5802
Genes= 20 | added=ENSG00000103512      | val_AUC=0.9755 | test_AUC=0.9408 | thr=0.6767
Genes= 21 | added=ENSG00000198522      | val_AUC=0.9742 | test_AUC=0.9435 | thr=0.6380
Genes= 22 | added=ENSG00000116120      | va